# Governança — Funções de máscara de PII (Unity Catalog)

Cria/atualiza as **funções reutilizáveis** de column mask no Unity Catalog
(`mask_cpf`, `mask_email`, `mask_name`, `mask_data_nascimento`). O valor em claro fica
disponível apenas para o grupo privilegiado (Entra ID). Usa a lib `security`.

Execute **uma vez**, e **antes** das dimensões: cada notebook de dimensão aplica as
máscaras às suas próprias colunas (as funções precisam já existir).

## Parâmetros e setup

In [ ]:
import sys

sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("privileged_group", "dm_pii_readers")

CATALOG    = dbutils.widgets.get("catalog")
SCHEMA     = dbutils.widgets.get("bronze_schema")
PRIV_GROUP = dbutils.widgets.get("privileged_group")

from security import column_mask_functions_sql

print("Catálogo:", CATALOG, "| Schema:", SCHEMA, "| Grupo PII:", PRIV_GROUP)

## Cria as funções de máscara reutilizáveis

In [ ]:
def run_sql_script(sql_text: str):
    for stmt in sql_text.split(";"):
        s = stmt.strip()
        if s:
            spark.sql(s)


sql_funcs = column_mask_functions_sql(CATALOG, SCHEMA, privileged_group=PRIV_GROUP)
print(sql_funcs)
run_sql_script(sql_funcs)
print("[OK] Funções de máscara criadas. Cada dimensão aplica as máscaras às suas colunas.")